# Sparse, Dense & Hybrid Retrieval with Pyversity

Pyversity is **embedding-agnostic**: its diversification algorithms only care about pairwise similarities between vectors, not how those vectors were produced. This means it works equally well with:

- **Sparse** vectors from BM25 (keyword-based term frequencies)
- **Dense** vectors from fast static models like [potion-base-32M](https://huggingface.co/minishlab/potion-base-32M)
- **Hybrid** combinations of both

This notebook demonstrates all three paradigms on the **Quora question corpus** (515k real questions from the Quora platform). We search for `"how to learn machine learning"` — a query that returns many near-identical question phrasings without diversification, and distinct subtopics with it.

In [1]:
%pip install pyversity bm25s sentence-transformers datasets


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import scipy.sparse as sp
import bm25s
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pyversity import diversify, Strategy

/Users/thomasvandongen/.pyenv/versions/3.12.8/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The Corpus

We use the **BEIR Quora corpus** — 515,000 real questions from the Quora platform, covering an enormous range of topics.

**Query:** `"what is machine learning"`

This query has a dramatic redundancy problem in the raw results: Quora users asked the exact same question dozens of times, so naive retrieval returns literal duplicates. Diversification collapses those duplicates and surfaces genuinely distinct angles — definitions, algorithms, comparisons, career questions, project ideas.

In [3]:
# Load the Quora corpus from BEIR (~15 MB, cached automatically after first download)
# 515,000 real questions from the Quora platform.
print("Downloading Quora corpus...")
ds = load_dataset("BeIR/quora", "corpus", split="corpus")

corpus = [
    (doc["title"] + " " + doc["text"]).strip()
    for doc in ds
    if len((doc["title"] + " " + doc["text"]).strip()) > 20
]

query = "what is machine learning"
K = 20   # candidates to retrieve
k = 5    # items to select after diversification

print(f"Corpus size : {len(corpus):,} questions")
print(f"Query       : '{query}'")
print(f"Retrieve top-{K}, then diversify to {k}")

Corpus size : 514,930 questions
Query       : 'what is machine learning'
Retrieve top-20, then diversify to 5


---
## Part 1: Sparse Retrieval with BM25

BM25 is a classic keyword-matching algorithm that scores documents based on term frequency and inverse document frequency, producing **sparse** term vectors — most entries are zero because most words do not appear in a given document.

Pyversity only needs an array of shape `(n_candidates, n_features)`. For BM25, we reconstruct the internal score matrix and slice to the top-K candidates. A single `.toarray()` call converts the scipy sparse matrix to a dense NumPy array that pyversity can consume.

> The primary point of this section is the API bridge: pyversity works with any NumPy array, sparse or dense. The diversity effect from BM25 term vectors is modest — they capture vocabulary overlap rather than semantic similarity, so they are a coarser signal for redundancy than dense embeddings.

In [4]:
# Index the full 515k-question corpus with BM25
corpus_tokens = bm25s.tokenize(corpus)
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

# Retrieve top-K candidates
query_tokens = bm25s.tokenize([query])
results, scores = retriever.retrieve(query_tokens, k=K)

bm25_candidate_indices = results[0].tolist()   # top-K indices into corpus
bm25_candidate_scores  = scores[0]             # BM25 relevance scores

print("Top-K BM25 candidates:")
for rank, (idx, score) in enumerate(zip(bm25_candidate_indices, bm25_candidate_scores), 1):
    print(f"  {rank:2d}. (score={score:.2f}) {corpus[idx]}")

Split strings:   0%|          | 0/514930 [00:00<?, ?it/s]

Split strings:   5%|▌         | 26641/514930 [00:00<00:01, 266396.28it/s]

Split strings:  11%|█         | 57004/514930 [00:00<00:01, 288290.89it/s]

Split strings:  17%|█▋        | 85834/514930 [00:00<00:02, 204807.45it/s]

Split strings:  23%|██▎       | 116711/514930 [00:00<00:01, 237725.21it/s]

...

BM25S Compute Scores:  96%|█████████▌| 492631/514930 [00:02<00:00, 175807.63it/s]

BM25S Compute Scores:  99%|█████████▉| 510820/514930 [00:02<00:00, 177565.69it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Top-K BM25 candidates:
   1. (score=7.41) What is machine to machine learning?
   2. (score=6.93) What is the difference between machine learning and statistical machine learning?
   3. (score=6.90) What is the machine learning?
   4. (score=6.90) What is machine learning?
   5. (score=6.90) what is machine learning?
   6. (score=6.90) What is machine learning?
   7. (score=6.87) What is Learning to Rank in machine learning?
   8. (score=6.87) What are the Prerequisites for learning Machine Learning?
   9. (score=6.66) Machine Learning: What are some innovative machine learning project ideas ?
  10. (score=6.50) What are prerequisites to start learning Machine Learning?
  11. (score=6.50) What do I start with for learning machine learning?
  12. (score=6.41) What's new in Machine Learning?
  13. (score=6.41) What is the new machine learning?
  14. (score=6.41) What is epochs in machine learning?
  15. (score=6.41) What is the alternative to machine learning?
  16. (score=6.41) What is 

In [5]:
# bm25s stores its term scores as raw CSR/CSC arrays.
# We reassemble them into a scipy sparse matrix (docs × vocab),
# then slice to our K candidates and call .toarray() —
# the only step needed to bridge sparse BM25 vectors into pyversity.
s = retriever.scores
bm25_score_matrix = sp.csc_matrix(
    (np.array(s["data"]), np.array(s["indices"]), np.array(s["indptr"])),
    shape=(int(s["num_docs"]), len(s["indptr"]) - 1),
)  # shape: (n_docs, vocab_size)

bm25_candidate_embeddings = bm25_score_matrix[bm25_candidate_indices].toarray()  # (K, vocab_size)

print(f"Sparse BM25 matrix shape : {bm25_score_matrix.shape}")
print(f"Candidate embeddings     : {bm25_candidate_embeddings.shape}  (K × vocab_size)")
print(f"Sparsity                 : {(bm25_candidate_embeddings == 0).mean():.1%} zeros")

Sparse BM25 matrix shape : (514930, 85392)
Candidate embeddings     : (20, 85392)  (K × vocab_size)
Sparsity                 : 100.0% zeros


In [6]:
# Naive top-5: diversity=0.0 → pure relevance ranking, no diversification
naive_bm25 = diversify(
    embeddings=bm25_candidate_embeddings,
    scores=bm25_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.0,
)

print("=== BM25 — Naive top-5 (diversity=0.0) ===")
for rank, i in enumerate(naive_bm25.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== BM25 — Naive top-5 (diversity=0.0) ===
  1. What is machine to machine learning?
  2. What is the difference between machine learning and statistical machine learning?
  3. What is the machine learning?
  4. What is machine learning?
  5. what is machine learning?


In [7]:
# Diversified top-5 with DPP
diverse_bm25 = diversify(
    embeddings=bm25_candidate_embeddings,
    scores=bm25_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.7,
)

print("=== BM25 — Diversified top-5 (diversity=0.7) ===")
for rank, i in enumerate(diverse_bm25.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== BM25 — Diversified top-5 (diversity=0.7) ===
  1. What is machine to machine learning?
  2. What is the difference between machine learning and statistical machine learning?
  3. What are the Prerequisites for learning Machine Learning?
  4. What is Learning to Rank in machine learning?
  5. Machine Learning: What are some innovative machine learning project ideas ?


---
## Part 2: Dense Re-ranking with Static Embeddings

Dense embeddings capture **semantic meaning** — "optimization" and "minimization" end up close even without shared tokens. Here we re-score the same BM25 top-K candidates using a dense encoder, then diversify based on dense similarity.

We use [**potion-base-32M**](https://huggingface.co/minishlab/potion-base-32M), a fast static embedding model from the [model2vec](https://github.com/MinishLab/model2vec) family. It is orders of magnitude faster than transformer-based encoders (no GPU needed) while retaining strong retrieval quality.

> This 2-stage pattern — BM25 for fast first-stage retrieval, dense model for re-ranking — is common in production systems.

In [8]:
# Encode only the 20 BM25 candidates and the query with potion-base-32M.
# This 2-stage pattern (BM25 retrieval → dense re-ranking) is common in production.
# device="cpu" — static embedding models do not benefit from GPU/MPS
model = SentenceTransformer("minishlab/potion-base-32M", device="cpu")

bm25_candidate_texts       = [corpus[i] for i in bm25_candidate_indices]
query_embedding            = model.encode([query], normalize_embeddings=True)
dense_candidate_embeddings = model.encode(bm25_candidate_texts, normalize_embeddings=True)

# Cosine similarity (L2-normalised vectors → dot product = cosine sim)
dense_candidate_scores = (query_embedding @ dense_candidate_embeddings.T)[0]

print("BM25 candidates re-scored by dense model:")
for rank, i in enumerate(np.argsort(dense_candidate_scores)[::-1], 1):
    print(f"  {rank:2d}. (score={dense_candidate_scores[i]:.3f}) {corpus[bm25_candidate_indices[i]]}")

BM25 candidates re-scored by dense model:
   1. (score=0.994) What is machine learning?
   2. (score=0.994) What is machine learning?
   3. (score=0.994) what is machine learning?
   4. (score=0.994) What is the machine learning?
   5. (score=0.965) What is the new machine learning?
   6. (score=0.957) What's new in Machine Learning?
   7. (score=0.951) What is machine to machine learning?
   8. (score=0.863) What do I start with for learning machine learning?
   9. (score=0.838) What is Learning to Rank in machine learning?
  10. (score=0.835) What is machine learning algorithm?
  11. (score=0.834) Machine Learning: What are some innovative machine learning project ideas ?
  12. (score=0.825) What is the difference between machine learning and statistical machine learning?
  13. (score=0.821) What is the alternative to machine learning?
  14. (score=0.811) What are the Prerequisites for learning Machine Learning?
  15. (score=0.779) What are prerequisites to start learning Machine Lea

In [9]:
# Naive top-5: no diversification — all results are near-identical question phrasings
naive_dense = diversify(
    embeddings=dense_candidate_embeddings,
    scores=dense_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.0,
)

print("=== Dense — Naive top-5 (diversity=0.0) ===")
for rank, i in enumerate(naive_dense.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== Dense — Naive top-5 (diversity=0.0) ===
  1. What is machine learning?
  2. what is machine learning?
  3. What is machine learning?
  4. What is the machine learning?
  5. What is the new machine learning?


In [10]:
# Diversified top-5: distinct subtopics surface (economics, statistics, programming, etc.)
diverse_dense = diversify(
    embeddings=dense_candidate_embeddings,
    scores=dense_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.7,
)

print("=== Dense — Diversified top-5 (diversity=0.7) ===")
for rank, i in enumerate(diverse_dense.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== Dense — Diversified top-5 (diversity=0.7) ===
  1. What is machine learning?
  2. Machine Learning: What are some innovative machine learning project ideas ?
  3. What is machine learning algorithm?
  4. What is Learning to Rank in machine learning?
  5. What is the difference between machine learning and statistical machine learning?


In [11]:
# Side-by-side: the contrast between naive and diversified is immediately visible
diverse_dense_full = diversify(
    embeddings=dense_candidate_embeddings,
    scores=dense_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=1.0,
)

print("Naive (d=0.0)   — all 5 ask essentially the same thing:")
for rank, i in enumerate(naive_dense.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

print()
print("Diversified (d=0.7) — varied perspectives on learning ML:")
for rank, i in enumerate(diverse_dense.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

print()
print("Maximum diversity (d=1.0) — maximally distinct subtopics:")
for rank, i in enumerate(diverse_dense_full.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

Naive (d=0.0)   — all 5 ask essentially the same thing:
  1. What is machine learning?
  2. what is machine learning?
  3. What is machine learning?
  4. What is the machine learning?
  5. What is the new machine learning?

Diversified (d=0.7) — varied perspectives on learning ML:
  1. What is machine learning?
  2. Machine Learning: What are some innovative machine learning project ideas ?
  3. What is machine learning algorithm?
  4. What is Learning to Rank in machine learning?
  5. What is the difference between machine learning and statistical machine learning?

Maximum diversity (d=1.0) — maximally distinct subtopics:
  1. What is machine to machine learning?
  2. What is the difference between machine learning and deep learning?
  3. Is machine learning dying?
  4. Oxford machine learning?
  5. What is epochs in machine learning?


---
## Part 3: Hybrid Re-ranking

Hybrid retrieval fuses BM25 and dense scores. We normalise each distribution to `[0, 1]` and linearly interpolate, then diversify using dense embeddings.

Two separate concerns:
- **Hybrid score** (BM25 + dense) — determines *which* candidates rank highest. Fusing both signals can improve ranking: BM25 rewards exact keyword matches, dense rewards semantic neighbours.
- **Dense embeddings** for diversity — determines *how redundant* two candidates are. BM25 term vectors only reflect vocabulary overlap with the query, not passage-to-passage semantic similarity, so dense embeddings are always the right tool for the diversity step.

In [12]:
# Fuse BM25 and dense scores for the same K candidates
def min_max_normalize(x: np.ndarray) -> np.ndarray:
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

alpha = 0.5
hybrid_candidate_scores = (
    alpha * min_max_normalize(dense_candidate_scores)
    + (1 - alpha) * min_max_normalize(bm25_candidate_scores)
)

print("BM25 candidates re-scored by hybrid (BM25 + dense):")
for rank, i in enumerate(np.argsort(hybrid_candidate_scores)[::-1], 1):
    print(f"  {rank:2d}. (score={hybrid_candidate_scores[i]:.3f}) {corpus[bm25_candidate_indices[i]]}")

BM25 candidates re-scored by hybrid (BM25 + dense):
   1. (score=0.919) What is machine to machine learning?
   2. (score=0.793) What is machine learning?
   3. (score=0.793) What is machine learning?
   4. (score=0.793) what is machine learning?
   5. (score=0.793) What is the machine learning?
   6. (score=0.537) What is the new machine learning?
   7. (score=0.523) What's new in Machine Learning?
   8. (score=0.487) What is the difference between machine learning and statistical machine learning?
   9. (score=0.485) What is Learning to Rank in machine learning?
  10. (score=0.433) What are the Prerequisites for learning Machine Learning?
  11. (score=0.393) Machine Learning: What are some innovative machine learning project ideas ?
  12. (score=0.385) What do I start with for learning machine learning?
  13. (score=0.291) What is machine learning algorithm?
  14. (score=0.264) What is the alternative to machine learning?
  15. (score=0.225) What are prerequisites to start learning M

In [13]:
naive_hybrid = diversify(
    embeddings=dense_candidate_embeddings,
    scores=hybrid_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.0,
)

print("=== Hybrid — Naive top-5 (diversity=0.0) ===")
for rank, i in enumerate(naive_hybrid.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== Hybrid — Naive top-5 (diversity=0.0) ===
  1. What is machine to machine learning?
  2. What is machine learning?
  3. what is machine learning?
  4. What is machine learning?
  5. What is the machine learning?


In [14]:
diverse_hybrid = diversify(
    embeddings=dense_candidate_embeddings,
    scores=hybrid_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.7,
)

print("=== Hybrid — Diversified top-5 (diversity=0.7) ===")
for rank, i in enumerate(diverse_hybrid.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== Hybrid — Diversified top-5 (diversity=0.7) ===
  1. What is machine to machine learning?
  2. What is Learning to Rank in machine learning?
  3. What are the Prerequisites for learning Machine Learning?
  4. What is the difference between machine learning and statistical machine learning?
  5. Machine Learning: What are some innovative machine learning project ideas ?


---
## Part 4: Comparing the Three Approaches

Diversified top-5 across all three retrieval methods, for the same query on the same corpus.

In [15]:
print("Diversified top-5 across all three retrieval methods")
print("=" * 90)

for label, result in [
    ("BM25 (sparse)",   diverse_bm25),
    ("Dense (potion)",  diverse_dense),
    ("Hybrid",          diverse_hybrid),
]:
    print(f"\n{label}:")
    for rank, i in enumerate(result.indices, 1):
        print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

Diversified top-5 across all three retrieval methods

BM25 (sparse):
  1. What is machine to machine learning?
  2. What is the difference between machine learning and statistical machine learning?
  3. What are the Prerequisites for learning Machine Learning?
  4. What is Learning to Rank in machine learning?
  5. Machine Learning: What are some innovative machine learning project ideas ?

Dense (potion):
  1. What is machine learning?
  2. Machine Learning: What are some innovative machine learning project ideas ?
  3. What is machine learning algorithm?
  4. What is Learning to Rank in machine learning?
  5. What is the difference between machine learning and statistical machine learning?

Hybrid:
  1. What is machine to machine learning?
  2. What is Learning to Rank in machine learning?
  3. What are the Prerequisites for learning Machine Learning?
  4. What is the difference between machine learning and statistical machine learning?
  5. Machine Learning: What are some innovative

---
## Key Takeaways

1. **Pyversity is embedding-agnostic.** DPP only operates on pairwise similarities between vectors — it does not matter whether those vectors came from BM25, a static model, a transformer, or anything else that produces a NumPy array.

2. **`.toarray()` is all you need for sparse vectors.** BM25 score matrices are scipy sparse arrays. One call converts them to a dense NumPy array that pyversity accepts.

3. **Dense embeddings unlock richer diversity.** Semantic similarity between candidates (not just keyword overlap) gives diversification a stronger signal, surfacing genuinely distinct subtopics rather than just different phrasings.

4. **The `diversity` parameter is a dial.** `0.0` = pure relevance ranking. `1.0` = maximum topic coverage. Values of `0.5–0.8` give the best relevance/diversity trade-off for most production use cases.